In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 
mart_schema = get_mart_schema() 

# -------------------------------
# Create mart Table (Environment-aware)
# -------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{features_schema}.features_sales_timeseries AS

WITH base_sales AS (
    SELECT
        o.order_date,              -- ✅ FIXED
        s.product_id,
        s.sales AS total_sales
    FROM {catalog}.{gold_schema}.facts_sales s
    JOIN {catalog}.{gold_schema}.facts_orders o
        ON s.order_id = o.order_id
),

time_features AS (
    SELECT
        order_date,
        product_id,

        -- LAG features (no leakage)
        LAG(total_sales, 7) OVER (
            PARTITION BY product_id
            ORDER BY order_date
        ) AS lag_7_sales,

        LAG(total_sales, 30) OVER (
            PARTITION BY product_id
            ORDER BY order_date
        ) AS lag_30_sales,

        -- Rolling averages (exclude current row)
        AVG(total_sales) OVER (
            PARTITION BY product_id
            ORDER BY order_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS rolling_avg_7,

        AVG(total_sales) OVER (
            PARTITION BY product_id
            ORDER BY order_date
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
        ) AS rolling_avg_30,

        -- Calendar features
        DAYOFWEEK(order_date) AS day_of_week,
        MONTH(order_date) AS month

    FROM base_sales
)

SELECT * FROM time_features;
""")

